In [ ]:
import pandas as pd
import yfinance as yf
from google.colab import files

# ====== CONFIG ======
TICKERS = ["AAPL", "NVDA", "GOOGL"]
START_DATE = "2018-01-01"
END_DATE   = "2025-12-31"

FACT_CSV = "fact_prices_daily.csv"
DIM_SYMBOL_CSV = "dim_symbol.csv"
# =====================

def main():
    all_rows = []

    for tkr in TICKERS:
        df = yf.download(
            tkr,
            start=START_DATE,
            end=END_DATE,
            auto_adjust=True,   # כל המחירים מתואמים (Open/High/Low/Close)
            progress=False
        )

        if df is None or df.empty:
            print(f"⚠️ אין נתונים עבור {tkr}")
            continue

        # אם הגיע MultiIndex
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df = df.reset_index().rename(columns={"Date": "date"})
        df["symbol"] = tkr

        required = {"Open", "High", "Low", "Close", "Volume"}
        if not required.issubset(set(df.columns)):
            raise ValueError(f"חסרות עמודות עבור {tkr}. יש: {list(df.columns)}")

        # המרות וניקוי
        df["open_price"] = pd.to_numeric(df["Open"], errors="coerce")
        df["high_price"] = pd.to_numeric(df["High"], errors="coerce")
        df["low_price"]  = pd.to_numeric(df["Low"], errors="coerce")
        df["adj_close"]  = pd.to_numeric(df["Close"], errors="coerce")  # מתואם בגלל auto_adjust=True
        df["volume"]     = pd.to_numeric(df["Volume"], errors="coerce").fillna(0)

        df = df.sort_values("date")

        # prev_close לחישובי תשואות
        df["prev_close"] = df["adj_close"].shift(1)

        # תשואה יומית "רגילה" (Close מול Close)
        df["stock_return"] = (df["adj_close"] - df["prev_close"]) / df["prev_close"]

        # תשואה יומית מקסימלית (High מול Close של אתמול)
        df["daily_return_high"] = (df["high_price"] - df["prev_close"]) / df["prev_close"]

        # תשואה יומית מינימלית (Low מול Close של אתמול)
        df["daily_return_low"] = (df["low_price"] - df["prev_close"]) / df["prev_close"]

        # להסיר יום ראשון לכל מניה (כי אין prev_close)
        df = df.dropna(subset=["prev_close", "stock_return", "daily_return_high", "daily_return_low"])

        # לבחור עמודות לטבלת העובדות
        df = df[[
            "date",
            "symbol",
            "open_price",
            "adj_close",
            "volume",
            "stock_return",
            "daily_return_high",
            "daily_return_low"
        ]]

        all_rows.append(df)

    fact = pd.concat(all_rows, ignore_index=True).sort_values(["symbol", "date"])

    # Fact table
    fact.to_csv(FACT_CSV, index=False, encoding="utf-8-sig")
    print(f"Saved: {FACT_CSV} | rows={len(fact)} | cols={len(fact.columns)}")

    # Dimension table: symbols
    dim_symbol = pd.DataFrame({"symbol": sorted(fact["symbol"].unique())})
    dim_symbol["asset_type"] = "stock"
    dim_symbol.to_csv(DIM_SYMBOL_CSV, index=False, encoding="utf-8-sig")
    print(f"Saved: {DIM_SYMBOL_CSV} | rows={len(dim_symbol)}")

    # הורדה ב-Colab
    files.download(FACT_CSV)
    files.download(DIM_SYMBOL_CSV)

if __name__ == "__main__":
    main()





Saved: fact_prices_daily.csv | rows=6027 | cols=8
Saved: dim_symbol.csv | rows=3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>